In [1]:
from collections import defaultdict
import os
import pickle
import re
import sys
import numpy as np
import pandas as pd

In [2]:
data_dir = '../data/'

comp_data = pickle.load(open(os.path.join(data_dir, 'prop_train_data_with_units.pkl'), 'rb')) #+ \
            #pickle.load(open(os.path.join(val_test_table_dir, 'all_val_test_data.pkl'), 'rb'))

full_comp_data = pickle.load(open(os.path.join(data_dir, 'train_data.pkl'), 'rb')) #+ \
            #pickle.load(open(os.path.join(val_test_table_dir, 'val_test_data.pkl'), 'rb'))

# comp_data = pickle.load(open(os.path.join(table_dir, 'train_data.pkl'), 'rb')) + \
#             pickle.load(open(os.path.join(table_dir, 'val_test_data.pkl'), 'rb'))
# comp_train_data = pickle.load(open(os.path.join(table_dir, 'val_test_data.pkl'), 'rb'))
comp_data_dict = {(c['pii'], c['t_idx']): c for c in comp_data}
full_comp_data_dict = {(c['pii'], c['t_idx']): c for c in full_comp_data}
old_train_val_test_split = pickle.load(open(os.path.join(data_dir, 'train_val_test_split.pkl'), 'rb'))
splits = ['train', 'val', 'test']

In [3]:
pii_tables = defaultdict(list)
anno_types = defaultdict(set)
for pii_tid in comp_data_dict:
    pii_tables[pii_tid[0]].append(comp_data_dict[pii_tid])
    anno_types[pii_tid[0]].add(comp_data_dict[pii_tid]['anno_type'])
full_pii_tables = defaultdict(list)
for pii_tid in full_comp_data_dict:
    full_pii_tables[pii_tid[0]].append(full_comp_data_dict[pii_tid])

In [4]:
# pick all tables from any paper that contains atleast one prop table
count_prop = 0
count_non_prop = 0
for pii in full_pii_tables:
    if pii in pii_tables:
        all_tables = full_pii_tables[pii]
        for table in all_tables:
            pii, tid = table['pii'], table['t_idx']
            if not (pii, tid) in comp_data_dict:
                table['prop_table'] = False
                if 'row_label' not in table.keys():
                    table['row_label'] = [0 for _ in range(table['num_rows'])]
                if 'col_label' not in table.keys():
                    table['col_label'] = [0 for _ in range(table['num_cols'])]
                table['anno_type'] = anno_types[pii]
                comp_data_dict[(pii, tid)] = table
                count_non_prop += 1
            else:
                count_prop += 1
                comp_data_dict[(pii, tid)]['anno_type'] = anno_types[pii]

In [5]:
print(count_prop)
print(count_non_prop)
print(len(comp_data_dict))

1021
988
2009


In [6]:
train_val_test_split = {'train': [], 'val': [], 'test': []}
for split in ['train', 'val', 'test']:
    print('split', split)
    for pii, tid in old_train_val_test_split[split]:
        if pii in pii_tables:
            train_val_test_split[split].append((pii, tid))

split train
split val
split test


In [7]:
print(len(train_val_test_split['train']))
print(len(train_val_test_split['val']))
print(len(train_val_test_split['test']))

2009
0
0


In [8]:
for split in ['train', 'val', 'test']:
    print()
    print(split)
    cp = 0
    cnp = 0
    for pii, tid in train_val_test_split[split]:
        t = comp_data_dict[(pii, tid)]
        if t['prop_table']:
            cp += 1
        elif not t['prop_table']:
            cnp += 1
        else:
            assert False
    print('cp', cp)
    print('cnp', cnp)


train
cp 1021
cnp 988

val
cp 0
cnp 0

test
cp 0
cnp 0


In [9]:
# all_train_data = [table for pii_tid, table in comp_data_dict if pii_tid in train_val_test_split['train']]
all_data = [table for pii_tid, table in comp_data_dict.items()]
all_train_data = []
all_val_test_data = []
for table in all_data:
    if (table['pii'], table['t_idx']) in train_val_test_split['train']:
        all_train_data.append(table)
    elif (table['pii'], table['t_idx']) in train_val_test_split['val']+train_val_test_split['test']:
        all_val_test_data.append(table)
    else:
        assert False
print(len(all_train_data))
print(len(all_val_test_data))

2009
0


In [10]:
pickle.dump(all_train_data, open(os.path.join(data_dir, 'all_train_data_pre_tuple_new.pkl'), 'wb'))

In [11]:
# al = {'nsns'}
# print(type(al))